# Valid JSON, wrong answer

**Scenario:** an anti money laundering pipeline turns analyst notes into case records. Strict schema
checking passes on every one. Then an audit finds a case whose review period ends before it starts,
carrying a negative amount, scored above the top of the scale.

Think of a schema as **a spell checker**. Every word is a real word, and the sentence still says
something nobody meant.

## Mechanics

A schema is the written shape of what is allowed. It has no opinion about whether 118 is a sensible
risk score. Most teams ship only the first layer.

| Check | Where it runs | What it rejects |
|---|---|---|
| `response_format` with `strict` | the provider, before you see the reply | a missing key, a wrong type, an extra key |
| `Field(ge=0, le=100)` | pydantic, per field | a number outside the range you allow |
| `@field_validator("name")` | pydantic, per field, once the type is settled | a value the type allows and the business does not |
| `@model_validator(mode="after")` | pydantic, once, on the finished record | two fields that disagree with each other |
| `ValidationError.errors()` | pydantic, on failure | a list, each entry naming `loc`, `type` and `msg` |

A validator is code that rejects a value that is the right shape and the wrong answer. Only the last
three rows do that.

## The picture

![Two gates. One reads types, the other reads meaning](images/structural-then-semantic.svg)

A record that clears the first gate has not cleared the second. Skip the second and the queue fills
with records that look like good ones.

## The cost

```
exposure = bad records shipped x (analyst rework + the cost of a late filing)
```

A bad record does not fail. It sits in the case queue looking like a good one, and an auditor finds it
rather than you.

## The failure

The note as the case management system stored it. Free text, unchecked on the way in, and three of its
numbers are wrong. That is ordinary.

In [1]:
NOTE = ("Case AML-2291. Analyst note, copied from the case management system. "
        "Counterparty wired 84,000 EUR through three shell entities. The reversal "
        "booked as -84000 EUR on the ledger. Review period runs from 2026-03-14 to "
        "2026-02-28. Analyst keyed the risk score as 118 on our 0 to 100 scale. "
        "Recommend escalation.")

SYSTEM = "You extract anti money laundering case reviews into JSON from analyst notes."

FIELDS = {"case_id": {"type": "string"},
          "risk_score": {"type": "integer"},
          "amount_eur": {"type": "number"},
          "review_from": {"type": "string"},
          "review_to": {"type": "string"},
          "recommendation": {"type": "string", "enum": ["close", "escalate", "file_sar"]}}

`strict` means the provider will not hand back anything that breaks the shape, so the only failures
left are failures of meaning.

In [2]:
import json
from vault import get_client, load_env, model_for

load_env()
client = get_client("09-programmatic-guardrails/01-valid-json-wrong-answer")

SCHEMA = {"type": "object", "properties": FIELDS, "required": list(FIELDS),
          "additionalProperties": False}
FORMAT = {"type": "json_schema",
          "json_schema": {"name": "case_review", "strict": True, "schema": SCHEMA}}


def ask_once():
    """One extraction. The shape is guaranteed. Nothing else is."""
    reply = client.chat.completions.create(
        model=model_for("default"), max_tokens=300, response_format=FORMAT,
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": NOTE}])
    return json.loads(reply.choices[0].message.content)

Once is an anecdote. Ask six times and count, because the question is not whether this can happen but
how often it does.

In [3]:
records = [ask_once() for _ in range(6)]

for i, r in enumerate(records):
    keys_ok = set(r) == set(FIELDS)
    print(f"  {i}  keys_ok={keys_ok}  score={r['risk_score']:>4}  "
          f"amount={r['amount_eur']:>10}  {r['review_from']} to {r['review_to']}")

in_range = sum(0 <= r["risk_score"] <= 100 for r in records)
print(f"\nevery record parsed and matched the schema")
assert in_range == len(records), f"{len(records) - in_range} of {len(records)} scored out of range"

  0  keys_ok=True  score= 118  amount=    -84000  2026-03-14 to 2026-02-28
  1  keys_ok=True  score= 118  amount=   84000.0  2026-03-14 to 2026-02-28
  2  keys_ok=True  score= 118  amount=     84000  2026-02-28 to 2026-03-14
  3  keys_ok=True  score= 118  amount=  -84000.0  2026-02-28 to 2026-03-14
  4  keys_ok=True  score= 118  amount=  -84000.0  2026-03-14 to 2026-02-28
  5  keys_ok=True  score= 118  amount=  -84000.0  2026-02-28 to 2026-03-14

every record parsed and matched the schema


AssertionError: 6 of 6 scored out of range

## The diagnosis

None of these are shape problems, so the strict schema had nothing to say. Six of six scored 118.
Four of six carried the negative amount.

**The score sits above the top of the scale.** To the schema, an integer is an integer.

**The amount is negative.** It is a ledger reversal, and the field means the sum under review.

**The dates run backwards.** The note itself is wrong. Three replies copied it. The other three
swapped the dates into a sensible order.

That split is the dangerous one. Half the time it repaired the source data without saying so.

## The fix

Ranges first, because they are cheap and they catch two of the three. `Field` puts the bound on the
type itself, so nothing constructs the object without it.

In [4]:
import datetime as dt
from pydantic import BaseModel, Field, ValidationError, field_validator, model_validator


class CaseReview(BaseModel):
    """Ranges only. Still not enough, and worth seeing why."""

    case_id: str = Field(pattern=r"^AML-\d{4}$")
    risk_score: int = Field(ge=0, le=100)
    amount_eur: float = Field(gt=0)
    review_from: dt.date
    review_to: dt.date
    recommendation: str

A range cannot see two fields at once, so the backwards period walks through it. That is the job of a
model validator, which runs once on the finished record.

In [5]:
class CheckedReview(CaseReview):
    """Adds the rules that need more than one field to see."""

    @model_validator(mode="after")
    def period_runs_forwards(self):
        if self.review_to < self.review_from:
            raise ValueError("review_to is before review_from")
        return self

    @field_validator("recommendation")
    @classmethod
    def known_action(cls, value):
        if value not in {"close", "escalate", "file_sar"}:
            raise ValueError(f"unknown recommendation {value!r}")
        return value

Now count. Ranges reject every record here, which hides what the cross-field rule adds, so the third
line repairs the two field problems by hand.

In [6]:
def accepted(model, rows):
    """How many of these records the model is willing to build."""
    kept = 0
    for row in rows:
        try:
            model(**row)
            kept += 1
        except ValidationError:
            pass
    return kept


repaired = [dict(r, risk_score=61, amount_eur=abs(r["amount_eur"])) for r in records]
n = len(records)
print(f"schema only, raw      : {n} of {n} accepted")
print(f"ranges, raw           : {accepted(CaseReview, records)} of {n} accepted")
print(f"ranges, fields fixed  : {accepted(CaseReview, repaired)} of {n} accepted")
print(f"cross-field, same rows: {accepted(CheckedReview, repaired)} of {n} accepted")

schema only, raw      : 6 of 6 accepted
ranges, raw           : 0 of 6 accepted
ranges, fields fixed  : 6 of 6 accepted
cross-field, same rows: 3 of 6 accepted


Rejecting is half the job. A case worker needs the field and the reason, so quarantine the record
rather than dropping it.

In [7]:
def quarantine(row):
    """Park a rejected record with the field and the reason, for a human."""
    try:
        return {"ok": True, "record": CheckedReview(**row)}
    except ValidationError as exc:
        reasons = [f"{e['loc'][0]}: {e['msg']}" for e in exc.errors()]
        return {"ok": False, "row": row, "reasons": reasons}


outcome = quarantine(records[0])
print(f"accepted: {outcome['ok']}")
for reason in outcome.get("reasons", []):
    print(f"  {reason}")

accepted: False
  risk_score: Input should be less than or equal to 100
  amount_eur: Input should be greater than 0


## The gate

The regression to stop is a bound relaxed to make an ingest run green. Two records on purpose, because
pydantic never reaches the model validator while a field is failing.

In [8]:
def test_impossible_values_are_rejected():
    bad = {"case_id": "AML-2291", "risk_score": 118, "amount_eur": -84000.0,
           "review_from": "2026-03-14", "review_to": "2026-02-28",
           "recommendation": "escalate"}
    clean_fields = dict(bad, risk_score=61, amount_eur=84000.0)
    for row, field in ((bad, "risk_score"), (bad, "amount_eur"),
                       (clean_fields, "review_to")):
        try:
            CheckedReview(**row)
        except ValidationError as exc:
            assert field in str(exc), f"{field} was not the reason"
            continue
        raise AssertionError(f"a record with a bad {field} was accepted")


test_impossible_values_are_rejected()
print("gate holds: score, amount and period are each rejected by name")

gate holds: score, amount and period are each rejected by name


Widen `le=100` to `le=1000` and the first case walks through.

### Enterprise exploration

- Where do these rules live when four services write to the same case queue?
- A quarantined case is a case nobody is working. What is the compliance cost of a week on hold?
- The model repaired the source data half the time. How would you measure that in production?

### Key takeaways

- A schema checks shape. It has no opinion about meaning.
- `Field` bounds catch single-field nonsense. Cross-field rules need a model validator.
- Reject with the field and the reason, or a human cannot act on it.
- Measure the rate. One run is not the behaviour.